In [ ]:
# In Jupyter Notebook
import sys
import os

sys.path.append(r"C:/Users/Phong/OneDrive - ICB Construction/Phong/data/Python_ETL/DS/ML_Models")
from experiments.run_model import run_experiment

# Run experiment
run_experiment(r"C:/Users/Phong/OneDrive - ICB Construction/Phong/data/Python_ETL/DS/ML_Models/configs/lightgbm_config.yaml")

In [ ]:
# In Jupyter Notebook
import sys
import os

# Set project path
project_path = r"C:/Users/Phong/OneDrive - ICB Construction/Phong/data/Python_ETL/DS/ML_Models"

# Change working directory to project folder
os.chdir(project_path)

# Add to path
sys.path.append(project_path)

from experiments.run_model import run_experiment

# Now relative path works
run_experiment("configs/xgboost_config.yaml")

In [ ]:
# In Jupyter cell
%run predictions/run_prediction.py --model models/catboost_model.pkl --config configs/catboost_config.yaml --data "C:/Users/Phong/OneDrive - ICB Construction/Phong/data/Python_ETL/DS/Database/Jobs to be Predicted.xlsx" --output "predictions/catboost_jobs_results.xlsx"

In [2]:
import pandas as pd

job_data_path = r"C:\Users\Phong\OneDrive - ICB Construction\Phong\data\Python_ETL\DS\ML_Models\data\Jobs Data.xlsx"

job_data_df = pd.read_excel(job_data_path)
job_data_df.describe()
job_data_df.columns


Index(['Job_Number', 'Job_Value', 'Total_Claimed', 'Total_Cost', 'Estimator',
       'Foreman', 'Job_Area', 'Main_Contractor', 'Suburb', 'Supervisor',
       'Job_Description', 'Saving_or_Loss'],
      dtype='object')

In [ ]:
# Linear Regression

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
import numpy as np
import pandas as pd

# Split data into features and target
X = job_data_df.drop(['Estimator',
                      'Foreman', 
                      'Job_Area', 
                      'Main_Contractor', 
                      'Suburb', 
                      'Supervisor',
                      'Job_Description',
                      'Saving_or_Loss'], 
                      axis=1)

y = job_data_df['Saving_or_Loss']

# Train - Test data split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Create and Train Model
model = LinearRegression()
model.fit(X_train, y_train)

# Predict
Y_pred = model.predict(X_test)

# Print Coefficients
print(f"Intercept: {model.intercept_}")
print(f"Slope: {model.coef_}")

# Evaluate
print(f"R² Score: {r2_score(y_test, Y_pred):.4f}")
print(f"RMSE: {np.sqrt(mean_squared_error(y_test, Y_pred)):.4f}")
print(f"MAE: {mean_absolute_error(y_test, Y_pred):.4f}")

Z_pred = model.predict([[10000, 2000, 3000, 40000]])
print(Z_pred)


In [ ]:
# Polynomial Regression - Without Pipeline and Tuning

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import make_pipeline
import numpy as np
import pandas as pd

# Split data into features and target
X = job_data_df.drop(['Estimator',
                      'Foreman', 
                      'Job_Area', 
                      'Main_Contractor', 
                      'Suburb', 
                      'Supervisor',
                      'Job_Description',
                      'Saving_or_Loss'], 
                      axis=1)

y = job_data_df['Saving_or_Loss']

# Train - Test data split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Create Polynomial features
degree = 2
poly = PolynomialFeatures(degree, include_bias=False)
X_train_poly = poly.fit_transform(X_train)
X_test_poly = poly.fit_transform(X_test)

# Train model on polynomial features
model_poly = LinearRegression()
model_poly.fit(X_train_poly, y_train)

# Predict
Y_pred = model_poly.predict(X_test_poly)

# Print Coefficients
print(f'Intercept: {model_poly.intercept_}')
print(f'Coefficients: {model_poly.coef_}')


Z_pred = model.predict([[10000, 2000, 3000, 40000]])
print(Z_pred)


In [ ]:
# Polynomial Regression - With Pipeline but no Tuning

from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.pipeline import Pipeline
import numpy as np
import pandas as pd

# Split data into features and target
X = job_data_df.drop(['Estimator',
                      'Foreman', 
                      'Job_Area', 
                      'Main_Contractor', 
                      'Suburb', 
                      'Supervisor',
                      'Job_Description',
                      'Saving_or_Loss'], 
                      axis=1)

y = job_data_df['Saving_or_Loss']

# Train - Test data split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Create conprehensive pipeline
pipeline = Pipeline([
    # Step 1: Handle missing values
    ('imputer', SimpleImputer(strategy='median')),
    # Step 2: Scale features
    ('scaler', StandardScaler()),
    # Step 3: Create polynomial features
    ('poly', PolynomialFeatures(degree=2, include_bias=False)),
    # Step 4: Final Estimator
    ('model', LinearRegression())
])

# Train model 
pipeline.fit(X_train, y_train)

# Predict
Y_pred = pipeline.predict(X_test)

# Model performance
score = pipeline.score(X_test, y_test)
print(f'R2 Score: {score}')

# Get parameters
params = pipeline.get_params()
print(params['poly__degree'])  # Access nested parameters

# Print Coefficients
print(f'Intercept: {pipeline.named_steps["model"].intercept_}')
print(f'Coefficients shape: {pipeline.named_steps["model"].coef_.shape}')
print(f'Coefficients: {pipeline.named_steps["model"].coef_}')


Z_pred = pipeline.predict([[10000, 2000, 3000, 40000]])
print(Z_pred)


In [ ]:
# Polynomial Regression - With Pipeline and Tuning

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np
import pandas as pd

# 1, Split data into features and target
X = job_data_df.drop(['Estimator',
                      'Foreman', 
                      'Job_Area', 
                      'Main_Contractor', 
                      'Suburb', 
                      'Supervisor',
                      'Job_Description',
                      'Saving_or_Loss'], 
                      axis=1)

y = job_data_df['Saving_or_Loss']

# 2, Train - Test data split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 3, Create pipeline with scaling and polynomial features + linear regression
pipeline = Pipeline([
    ('scaler', StandardScaler()),  # Important for polynomial features
    ('poly', PolynomialFeatures()),
    ('model', LinearRegression())
])

# 4, Define parameter grid
param_grid = {
    'poly__degree': [1, 2, 3, 4, 5],  # Try different polynomial degrees
    'poly__include_bias': [True, False],  # Include intercept term
    'poly__interaction_only': [True, False]  # Only interaction terms vs all terms
}

# 5, Perform grid search with cross-validation
grid_search = GridSearchCV(
    pipeline, 
    param_grid,
    cv=5,           # 5-fold cross-validation
    scoring='r2',   # Optimize for R-squared
    n_jobs=-1,      # Use all CPU cores
    verbose=1
)

print("performing Grid Search for best Polynomial parameter ...")

# 6, Train model
grid_search.fit(X_train, y_train)

  # Best parameters
print("\n" + "="*50)
print("BEST PARAMETERS FOUND:")
print("="*50)
print(f"Best Degree: {grid_search.best_params_['poly__degree']}")
print(f"Best include_bias: {grid_search.best_params_['poly__include_bias']}")
print(f"Best interaction_only: {grid_search.best_params_['poly__interaction_only']}")
print(f"Best CV R² Score: {grid_search.best_score_:.4f}")

  # Best model
best_poly_model = grid_search.best_estimator_

# 7, Predictions
y_pred = best_poly_model.predict(X_test)

# 8, Evaluate
print("\n" + "="*50)
print("MODEL EVALUATION ON TEST SET:")
print("="*50)
print(f"R² Score: {r2_score(y_test, y_pred):.4f}")
print(f"RMSE: {np.sqrt(mean_squared_error(y_test, y_pred)):.4f}")
print(f"MAE: {mean_absolute_error(y_test, y_pred):.4f}")

# 9, Usage
Z_pred = model.predict([[10000, 2000, 3000, 40000]])
print(Z_pred)


In [ ]:
# Polynomial Regression - With Pipeline and Tuning and Encode for Categorical Features

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.preprocessing import PolynomialFeatures, StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.impute import SimpleImputer
import numpy as np
import pandas as pd

# 1, Define which columns are numeric and which are categorical
numeric_features = ['Job_Number', 'Job_Value', 'Total_Claimed', 'Total_Cost']
categorical_features = ['Estimator', 'Foreman', 'Supervisor']

# 2, Prepare the data
X = job_data_df[numeric_features + categorical_features]
y = job_data_df['Saving_or_Loss']

# 3, Data Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 4, Create preprocessing for numeric columns
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# 5, Create preprocessing for categorical columns
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

# 6, Combine preprocessors
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ]
)

# 7, Create full pipeline with preprocessor, polu features and model
pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('poly', PolynomialFeatures()),
    ('model', Ridge())    
    # 'preprocessor__num__imputer__strategy': ['mean', 'median', 'most_frequent'] --For tuning imputer
])

# 8, Define parameter grid tuning
param_grid = {
    'poly__degree': [1, 2, 3, 4],
    #'poly__include_bias': [True, False],
    'poly__interaction_only': [True, False],
    'model__alpha': [0.1, 1.0, 10.0]  # Regularization strength
}

# 9, Perform grid search with cross-validation
grid_search = GridSearchCV(
    pipeline,
    param_grid,
    cv=5,
    scoring='r2',
    n_jobs=-1,
    verbose=1    
)

print("Performing Grid Search for the best parameters...")
grid_search.fit(X_train, y_train)


# 10. Best parameters
print("\n" + "="*70)
print("BEST PARAMETERS FOUND:")
print("="*70)
print(f"Best Degree: {grid_search.best_params_['poly__degree']}")
# print(f"Best include_bias: {grid_search.best_params_['poly__include_bias']}")
print(f"Best interaction_only: {grid_search.best_params_['poly__interaction_only']}")
print(f"Best CV R² Score: {grid_search.best_score_:.4f}")

# 11. Best model
best_model = grid_search.best_estimator_

# 12. Predictions
y_pred = best_model.predict(X_test)

# 13. Evaluate
print("\n" + "="*70)
print("MODEL EVALUATION ON TEST SET:")
print("="*70)
print(f"R² Score: {r2_score(y_test, y_pred):.4f}")
print(f"RMSE: {np.sqrt(mean_squared_error(y_test, y_pred)):.4f}")
print(f"MAE: {mean_absolute_error(y_test, y_pred):.4f}")

# 14. Get feature information
preprocessor_fitted = best_model.named_steps['preprocessor']
poly_fitted = best_model.named_steps['poly']

# Get feature names after preprocessing
feature_names = []
for name, transformer, columns in preprocessor_fitted.transformers_:
    if name == 'num':
        feature_names.extend(columns)
    elif name == 'cat':
        # Get one-hot encoded feature names
        encoder = transformer.named_steps['onehot']
        feature_names.extend(encoder.get_feature_names_out(columns))

# Get polynomial feature names
poly_feature_names = poly_fitted.get_feature_names_out(feature_names)
print(f"\nTotal features after polynomial transformation: {len(poly_feature_names)}")
print(f"First 10 polynomial features: {poly_feature_names[:10]}")

# 15. Make predictions for new data WITH categorical variables
# Create new job data with both numeric and categorical values
new_job = pd.DataFrame({
    'Job_Number': [10003],
    'Job_Value': [300000], 
    'Total_Claimed': [200000], 
    'Total_Cost': [150000],
    'Estimator': ['John Doe'],  # Categorical
    'Foreman': ['Jane Smith'],  # Categorical
    'Supervisor': ['Bob Johnson']  # Categorical
})

Z_pred = best_model.predict(new_job)
print(f"\nPrediction for new job: ${Z_pred[0]:,.2f}")

#Categorical + Polynomial = Problems - One-hot encoded columns create sparse matrices that blow up with polynomial features
#Use interaction_only=True - This avoids x² terms which cause the most issues


In [ ]:
# Regularized Polynomial Regression

from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.linear_model import Ridge, Lasso, ElasticNet, LinearRegression
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# Load and prepare data
X = job_data_df.drop(['Estimator', 'Foreman', 'Job_Area', 'Main_Contractor', 
                      'Suburb', 'Supervisor', 'Job_Description', 'Saving_or_Loss'], axis=1)
y = job_data_df['Saving_or_Loss']

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Create base pipeline (without regularization for comparison)
base_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
    ('poly', PolynomialFeatures(degree=2, include_bias=False)),
    ('model', LinearRegression())
])

# Ridge pipeline
ridge_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
    ('poly', PolynomialFeatures(degree=2, include_bias=False)),
    ('ridge', Ridge(alpha=1.0))
])

# Lasso pipeline
lasso_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
    ('poly', PolynomialFeatures(degree=2, include_bias=False)),
    ('lasso', Lasso(alpha=1.0, max_iter=10000))
])

# ElasticNet pipeline
elastic_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
    ('poly', PolynomialFeatures(degree=2, include_bias=False)),
    ('elastic', ElasticNet(alpha=1.0, l1_ratio=0.5, max_iter=10000))
])

# Train all models
base_pipeline.fit(X_train, y_train)
ridge_pipeline.fit(X_train, y_train)
lasso_pipeline.fit(X_train, y_train)
elastic_pipeline.fit(X_train, y_train)

# Compare results
print("="*60)
print("MODEL COMPARISON (Without Hyperparameter Tuning)")
print("="*60)

models = {
    'Linear Regression (No Reg)': base_pipeline,
    'Ridge (L2)': ridge_pipeline,
    'Lasso (L1)': lasso_pipeline,
    'ElasticNet (L1+L2)': elastic_pipeline
}

for name, model in models.items():
    y_pred = model.predict(X_test)
    r2 = r2_score(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    mae = mean_absolute_error(y_test, y_pred)
    
    # Get coefficient statistics
    if 'ridge' in model.named_steps:
        coef = model.named_steps['ridge'].coef_
    elif 'lasso' in model.named_steps:
        coef = model.named_steps['lasso'].coef_
    elif 'elastic' in model.named_steps:
        coef = model.named_steps['elastic'].coef_
    else:
        coef = model.named_steps['model'].coef_
    
    print(f"\n{name}:")
    print(f"  R² Score: {r2:.4f}")
    print(f"  RMSE: {rmse:.2f}")
    print(f"  MAE: {mae:.2f}")
    print(f"  Non-zero coeffs: {np.sum(coef != 0)} / {len(coef)}")
    print(f"  Max coefficient: {np.abs(coef).max():.2f}")
    print(f"  Mean coefficient: {np.abs(coef).mean():.2f}")

In [ ]:
# Regularized Polynomial Regression - Tuning

from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.linear_model import Ridge, Lasso, ElasticNet, LinearRegression
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# Load and prepare data
X = job_data_df.drop(['Estimator', 'Foreman', 'Job_Area', 'Main_Contractor', 
                      'Suburb', 'Supervisor', 'Job_Description', 'Saving_or_Loss'], axis=1)
y = job_data_df['Saving_or_Loss']

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Ridge pipeline
ridge_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
    ('poly', PolynomialFeatures(degree=2, include_bias=False)),
    ('ridge', Ridge(alpha=1.0))
])

# Lasso pipeline
lasso_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
    ('poly', PolynomialFeatures(degree=2, include_bias=False)),
    ('lasso', Lasso(alpha=1.0, max_iter=10000))
])

# ElasticNet pipeline
elastic_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
    ('poly', PolynomialFeatures(degree=2, include_bias=False)),
    ('elastic', ElasticNet(alpha=1.0, l1_ratio=0.5, max_iter=10000))
])

# Ridge Hyperparameter Tuning
print("\n" + "="*60)
print("RIDGE REGRESSION TUNING")
print("="*60)

ridge_param_grid = {
    'poly__degree': [1, 2, 3, 4],
    'ridge__alpha': [0.001, 0.01, 0.1, 1, 10, 100, 1000],
    'ridge__solver': ['auto', 'svd', 'cholesky', 'lsqr', 'sag']
}

ridge_search = GridSearchCV(
    ridge_pipeline, 
    ridge_param_grid, 
    cv=5, 
    scoring='r2',
    n_jobs=-1,
    verbose=0
)
ridge_search.fit(X_train, y_train)

print(f"Best Ridge Parameters: {ridge_search.best_params_}")
print(f"Best Ridge CV R²: {ridge_search.best_score_:.4f}")
print(f"Ridge Test R²: {ridge_search.score(X_test, y_test):.4f}")

# Lasso Hyperparameter Tuning
print("\n" + "="*60)
print("LASSO REGRESSION TUNING")
print("="*60)

lasso_param_grid = {
    'poly__degree': [1, 2, 3],
    'lasso__alpha': [0.0001, 0.001, 0.01, 0.1, 1, 10],
    'lasso__selection': ['cyclic', 'random']
}

lasso_search = GridSearchCV(
    lasso_pipeline, 
    lasso_param_grid, 
    cv=5, 
    scoring='r2',
    n_jobs=-1,
    verbose=0
)
lasso_search.fit(X_train, y_train)

print(f"Best Lasso Parameters: {lasso_search.best_params_}")
print(f"Best Lasso CV R²: {lasso_search.best_score_:.4f}")
print(f"Lasso Test R²: {lasso_search.score(X_test, y_test):.4f}")

# Count features selected by Lasso
best_lasso = lasso_search.best_estimator_
lasso_coef = best_lasso.named_steps['lasso'].coef_
print(f"Features selected by Lasso: {np.sum(lasso_coef != 0)} / {len(lasso_coef)}")

# ElasticNet Hyperparameter Tuning
print("\n" + "="*60)
print("ELASTICNET REGRESSION TUNING")
print("="*60)

elastic_param_grid = {
    'poly__degree': [1, 2, 3],
    'elastic__alpha': [0.001, 0.01, 0.1, 1, 10],
    'elastic__l1_ratio': [0.1, 0.3, 0.5, 0.7, 0.9, 1.0],  # 0=Ridge, 1=Lasso
    'elastic__selection': ['cyclic', 'random']
}

elastic_search = GridSearchCV(
    elastic_pipeline, 
    elastic_param_grid, 
    cv=5, 
    scoring='r2',
    n_jobs=-1,
    verbose=0
)
elastic_search.fit(X_train, y_train)

print(f"Best ElasticNet Parameters: {elastic_search.best_params_}")
print(f"Best ElasticNet CV R²: {elastic_search.best_score_:.4f}")
print(f"ElasticNet Test R²: {elastic_search.score(X_test, y_test):.4f}")

# Compare all tuned models
print("\n" + "="*60)
print("FINAL COMPARISON (Tuned Models)")
print("="*60)

tuned_models = {
    'Ridge (Tuned)': ridge_search,
    'Lasso (Tuned)': lasso_search,
    'ElasticNet (Tuned)': elastic_search
}

best_r2 = -np.inf
best_name = None
best_model = None

for name, model in tuned_models.items():
    test_r2 = model.score(X_test, y_test)
    print(f"\n{name}:")
    print(f"  Test R²: {test_r2:.4f}")
    print(f"  Best Params: {model.best_params_}")
    
    if test_r2 > best_r2:
        best_r2 = test_r2
        best_name = name
        best_model = model

print(f"\n{'='*60}")
print(f"🏆 BEST MODEL: {best_name} with R² = {best_r2:.4f}")
print("="*60)

In [ ]:
# Visualize the effect of alpha on coefficients
import matplotlib.pyplot as plt

alphas = [0.001, 0.01, 0.1, 1, 10, 100]
coefficient_paths = []

for alpha in alphas:
    ridge = Ridge(alpha=alpha)
    ridge_pipeline = Pipeline([
        ('scaler', StandardScaler()),
        ('poly', PolynomialFeatures(degree=2)),
        ('ridge', ridge)
    ])
    ridge_pipeline.fit(X_train, y_train)
    coefs = ridge_pipeline.named_steps['ridge'].coef_
    coefficient_paths.append(coefs)

coefficient_paths = np.array(coefficient_paths)

# Plot coefficient paths
plt.figure(figsize=(10, 6))
for i in range(min(10, coefficient_paths.shape[1])):  # Plot first 10 coefficients
    plt.plot(alphas, coefficient_paths[:, i], label=f'Feature {i}')

plt.xscale('log')
plt.xlabel('Alpha (regularization strength)')
plt.ylabel('Coefficient Value')
plt.title('Ridge Coefficient Paths\n(As alpha increases, coefficients shrink toward zero)')
plt.legend(loc='best', ncol=2, fontsize=8)
plt.grid(True, alpha=0.3)
plt.show()